In [1]:
# ============================================================
# CIFAR100 Dense + Sparse ViT (SAFE + FASTER VERSION)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy
import random
import numpy as np
import os

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# Seed
# ------------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
def get_cifar100(batch_size=64):

    transform_train = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3,(0.5,)*3)
    ])

    transform_test = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3,(0.5,)*3)
    ])

    trainset = torchvision.datasets.CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    testset = torchvision.datasets.CIFAR100(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

    trainloader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    return trainloader, testloader


train_dl, test_dl = get_cifar100()

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
@torch.no_grad()
def evaluate(model):

    model.eval()

    correct = 0
    total = 0

    for x, y in test_dl:

        x = x.to(device)
        y = y.to(device)

        out = model(x)

        pred = out.argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

    return correct / total


# ------------------------------------------------------------
# Sparse Wrapper
# ------------------------------------------------------------
class SparseViT(nn.Module):

    def __init__(self, base_model, keep_ratio=0.5):
        super().__init__()

        self.base = base_model
        self.keep_ratio = keep_ratio

    def forward(self, x):

        x = self.base._process_input(x)

        B, N, C = x.shape

        cls_token = self.base.class_token.expand(B, -1, -1)

        x = torch.cat((cls_token, x), dim=1)

        x = x + self.base.encoder.pos_embedding

        scores = x.norm(dim=-1)

        k = int(N * self.keep_ratio)

        topk = torch.topk(scores[:,1:],k,dim=1).indices + 1

        cls_idx = torch.zeros(B,1,dtype=torch.long,device=x.device)

        keep_idx = torch.cat([cls_idx,topk],dim=1)

        x = torch.gather(x,1,keep_idx.unsqueeze(-1).expand(-1,-1,C))

        x = self.base.encoder.layers(x)

        x = self.base.encoder.ln(x)

        cls = x[:,0]

        return self.base.heads(cls)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
def train(model, model_name, epochs=20):

    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

    scaler = torch.amp.GradScaler("cuda")

    best_acc = 0

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for x,y in train_dl:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):

                out = model(x)

                loss = F.cross_entropy(out,y)

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            total_loss += loss.item()

        val_acc = evaluate(model)

        print(f"Epoch {epoch+1}/{epochs} | Loss {total_loss/len(train_dl):.4f} | Val {val_acc:.4f}")

        # SAVE BEST MODEL
        if val_acc > best_acc:

            best_acc = val_acc

            save_path = f"{model_name}_best.pt"

            torch.save(model.state_dict(), save_path)

            print("Saved checkpoint:", save_path)

    print("Best accuracy:",best_acc)

    return model


# ============================================================
# TRAIN DENSE
# ============================================================

print("\nTraining Dense ViT")

dense = vit_b_16(weights="IMAGENET1K_V1")

dense.heads.head = nn.Linear(dense.heads.head.in_features,100)

dense = train(dense,"dense_cifar100",epochs=20)


# ============================================================
# TRAIN SPARSE
# ============================================================

print("\nTraining Sparse ViT (50%)")

sparse = SparseViT(copy.deepcopy(dense),keep_ratio=0.5)

sparse = train(sparse,"sparse_cifar100",epochs=20)


print("\nTraining finished")

print("\nSaved models should exist:")

os.system("ls *.pt")

Using device: cuda


100%|██████████| 169M/169M [00:05<00:00, 29.6MB/s] 



Training Dense ViT
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 211MB/s]  


Epoch 1/20 | Loss 1.2569 | Val 0.8402
Saved checkpoint: dense_cifar100_best.pt
Epoch 2/20 | Loss 0.3894 | Val 0.8559
Saved checkpoint: dense_cifar100_best.pt
Epoch 3/20 | Loss 0.2281 | Val 0.8615
Saved checkpoint: dense_cifar100_best.pt
Epoch 4/20 | Loss 0.1486 | Val 0.8620
Saved checkpoint: dense_cifar100_best.pt
Epoch 5/20 | Loss 0.1034 | Val 0.8641
Saved checkpoint: dense_cifar100_best.pt
Epoch 6/20 | Loss 0.0799 | Val 0.8690
Saved checkpoint: dense_cifar100_best.pt
Epoch 7/20 | Loss 0.0687 | Val 0.8696
Saved checkpoint: dense_cifar100_best.pt
Epoch 8/20 | Loss 0.0511 | Val 0.8592
Epoch 9/20 | Loss 0.0551 | Val 0.8659
Epoch 10/20 | Loss 0.0382 | Val 0.8577
Epoch 11/20 | Loss 0.0380 | Val 0.8661
Epoch 12/20 | Loss 0.0432 | Val 0.8671
Epoch 13/20 | Loss 0.0294 | Val 0.8612
Epoch 14/20 | Loss 0.0326 | Val 0.8703
Saved checkpoint: dense_cifar100_best.pt
Epoch 15/20 | Loss 0.0285 | Val 0.8655
Epoch 16/20 | Loss 0.0309 | Val 0.8639
Epoch 17/20 | Loss 0.0240 | Val 0.8688
Epoch 18/20 | Loss

0